<a href="https://www.kaggle.com/code/gastondana/imageto3d-benchmark-v1-execution?scriptVersionId=287691232" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# CODE BLOCK 1 & 2: FULL SETUP, CONFIGURATION, AND FAL GENERATION (MERGED AND FINALIZED)

# --- 1. Installation (Run this cell first on Kaggle) ---
!pip install --force-reinstall fal-client
!pip install pandas trimesh[easy] requests python-dotenv pillow tqdm

# --- 2. Imports ---
import os
import pandas as pd
import trimesh
import numpy as np
import requests
import json
from io import BytesIO
import time 
import base64 
import tempfile
from tqdm.notebook import tqdm 
from PIL import Image

# Import the client for reliable secret access
from kaggle_secrets import UserSecretsClient

# 🔑 Import the necessary components for FAL generation
from fal_client import submit, upload_file 

# --- 3. Configuration & Paths ---
# Use one consistent variable name for the environment check
ENV_KEY_NAME = "FAL_KEY"

# 🔑 FINAL FIX: Load API Key and set it as an Environment Variable
try:
    user_secrets = UserSecretsClient()
    # Assuming your secret is named 'FAL_API_KEY' in Kaggle Secrets
    api_key_value = user_secrets.get_secret("FAL_API_KEY") 
    
    # CRITICAL: Set the variable using the name the code checks for (FAL_KEY)
    os.environ[ENV_KEY_NAME] = api_key_value
    
    # Also set the Python variable for use in functions
    FAL_API_KEY = api_key_value 
    
    # Load Judge Key (assuming it's still needed later)
    JUDGE_API_KEY = user_secrets.get_secret("JUDGE_API_KEY") 

    print(f"FAL_API_KEY Status: ✅ Loaded and set to os.environ['{ENV_KEY_NAME}'].")
except Exception as e:
    FAL_API_KEY = None
    JUDGE_API_KEY = None
    print(f"FAL_API_KEY Status: ❌ ERROR loading secret. Error: {e}")

# Check if the key is actually available before proceeding (prevents crashes)
if not FAL_API_KEY:
    raise ValueError("❌ FATAL: FAL_API_KEY not loaded from Kaggle Secrets.")
print(f"✅ FAL API Key loaded: {FAL_API_KEY[:8]}...")

DATASET_ROOT = '/kaggle/input/aird-benchmark-assets/' 
OUTPUT_DIR = '/kaggle/working/results/' 
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 4. Load Manifest ---
CSV_PATH = os.path.join(DATASET_ROOT, 'benchmark_manifest.csv')
df = pd.read_csv(CSV_PATH, header=1) 
df = df.drop(df.index[0])
df = df[df['case_id'].astype(str).str.contains('S02_', na=False)]
df = df.reset_index(drop=True) 

# 🔑 DEFINITIVE PATH FIXES (From Code Block 1)
IMAGE_PATH_COL = 'image_file_path'
GLB_PATH_COL = 'ground_truth_3d_path' 
KAGGLE_SUBFOLDER = 'Data'

# 1. Clean and prefix the paths
if IMAGE_PATH_COL in df.columns:
    df[IMAGE_PATH_COL] = df[IMAGE_PATH_COL].astype(str).str.replace('data/', '', regex=False)
    df[IMAGE_PATH_COL] = KAGGLE_SUBFOLDER + '/' + df[IMAGE_PATH_COL]
if GLB_PATH_COL in df.columns: 
    df[GLB_PATH_COL] = df[GLB_PATH_COL].astype(str).str.replace('data/', '', regex=False)
    df[GLB_PATH_COL] = KAGGLE_SUBFOLDER + '/' + df[GLB_PATH_COL]
      
# 2. Final File Extension Fix
if IMAGE_PATH_COL in df.columns:
    df[IMAGE_PATH_COL] = (
        df[IMAGE_PATH_COL]
        .astype(str)
        .str.replace(r'\.(png|jpg|gif)$', '', regex=True)
        + '.jpeg'
    )

# --- Output and Column Initialization ---
new_cols = ['fal_generated_3d_path', 'fal_api_cost_usd', 'fal_api_token_count', 
            'chamfer_distance', 'hausdorff_distance', 'llm_judge_score_overall', 
            'llm_judge_rationale']
for col in new_cols:
    if col not in df.columns:
        df[col] = np.nan 

print(f"\nManifest loaded with {len(df)} cases.")
print("--- Setup Complete. Proceeding to FAL Generation Logic ---")

# ==============================================================================
# 🌟 START OF FAL GENERATION LOGIC (FROM FORMER CODE BLOCK 2) 🌟
# ==============================================================================

# --- CONFIGURATION ---
MODEL_ID = "fal-ai/trellis" 

# 🛠️ FINAL FIX: Simplified upload function to force disk upload via file path
def upload_image_to_fal(img, case_id):
    """
    Uploads image to FAL by saving it to a temporary file and passing the file path.
    """
    tmp_path = None
    
    try:
        # Prepare image 
        MAX_SIZE = (512, 512)
        img.thumbnail(MAX_SIZE, Image.Resampling.LANCZOS)
        
        print(f"  📤 Saving to temp file for upload...")
        
        # Save the binary image data to a temporary file on disk
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False, mode='wb') as tmp:
            img.save(tmp, format='JPEG', quality=95)
            tmp_path = tmp.name
        
        # Upload using the file path (string) - the reliable input type
        uploaded_url = upload_file(tmp_path)
        
        print(f"  ✅ Success via temp file!")
        return uploaded_url
    
    except Exception as e:
        # Re-raise the exception to be caught in run_fal_generation
        raise Exception(f"Failed to upload image via temp file for {case_id}: {e}")
    
    finally:
        # Clean up the temporary file
        if tmp_path and os.path.exists(tmp_path):
            os.unlink(tmp_path)
        

def run_fal_generation(row):
    """
    Complete workflow: Load image → Upload → Generate 3D → Download result
    """
    case_id = row['case_id']
    
    global FAL_API_KEY 
    
    print(f"\n{'='*60}")
    print(f"🎯 Processing Case: {case_id}")
    print(f"{'='*60}")
    
    # --- 1. LOAD AND PREPARE IMAGE ---
    image_path = os.path.join(DATASET_ROOT, row['image_file_path'])
    
    try:
        img = Image.open(image_path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
    except Exception as e:
        print(f"  ❌ FATAL: Cannot load image: {e}")
        return row
    
    # --- 2. UPLOAD IMAGE TO FAL STAGING ---
    try:
        uploaded_image_url = upload_image_to_fal(img, case_id)
        print(f"  🌐 Staged URL: {uploaded_image_url}")
    except Exception as e:
        print(f"  ❌ FATAL: Upload failed: {e}")
        return row
    
    # --- 3. SUBMIT 3D GENERATION JOB ---
    inputs = {
        "image_url": uploaded_image_url,
    }
    
    if pd.notna(row.get('llm_reverse_prompt')):
        inputs["prompt"] = row['llm_reverse_prompt']
        
    inputs.update({
        "seed": 42,
        "sparse_structure_steps": 12,
        "slat_steps": 12,
        "generate_model": True
    })
    
    try:
        print(f"\n🚀 Submitting to {MODEL_ID}...")
        # ⚠️ CRITICAL FIX: Removed 'key' argument, as it caused crash in fal-client 0.10.0
        job_handler = submit(
            MODEL_ID, 
            arguments=inputs
        )
        
        job_id = job_handler.request_id
        print(f"  📋 Job ID: {job_id}")
        print(f"  ⏳ Waiting for completion...")
        
        # Poll for result
        start_time = time.time()
        result = job_handler.get()
        elapsed = time.time() - start_time
        
        print(f"  ✅ Completed in {elapsed:.1f}s")
        
    except Exception as e:
        print(f"  ❌ FATAL: Job submission/polling failed: {e}")
        return row
    
    # --- 4. EXTRACT AND DOWNLOAD 3D MODEL ---
    glb_url = None
    
    # 🌟 NEW FIX: Check for the expected 'model_mesh' key from Trellis API output
    if 'model_mesh' in result and isinstance(result['model_mesh'], dict):
        glb_url = result['model_mesh'].get('url')
    
    # Fallback checks (less likely for current Trellis API)
    # The initial checks were:
    # if 'model' in result and isinstance(result['model'], dict):
    #     glb_url = result['model'].get('url')
    # elif 'glb_url' in result:
    #     glb_url = result['glb_url']
    # elif '3d_model_url' in result:
    #     glb_url = result['3d_model_url']
    
    if not glb_url:
        print(f"  ❌ No GLB URL found in result! Result keys found: {list(result.keys())}")
        return row
    
    print(f"  🎉 GLB URL found: {glb_url[:60]}...")
    
    # Download the 3D model
    try:
        generated_3d_path = os.path.join(OUTPUT_DIR, f"{case_id}_FAL_GEN.glb")
        glb_response = requests.get(glb_url, timeout=60)
        glb_response.raise_for_status()
        
        with open(generated_3d_path, 'wb') as f:
            f.write(glb_response.content)
            
        file_size = os.path.getsize(generated_3d_path) / 1024
        print(f"  ✅ Saved: {generated_3d_path} ({file_size:.1f} KB)")
        
        # Update the row with results
        row['fal_generated_3d_path'] = generated_3d_path
        row['fal_api_cost_usd'] = result.get('cost_usd', np.nan)
        row['fal_api_token_count'] = result.get('token_count', np.nan)
        
    except Exception as e:
        print(f"  ❌ FATAL: GLB download failed: {e}")
        return row
    
    print(f"\n✅ SUCCESS: {case_id} processed completely!")
    return row


# --- EXECUTION: TEST ON FIRST CASE ---
print("\n" + "="*60)
print("🧪 RUNNING TEST ON FIRST CASE (S02_001)")
print("="*60)

# The DataFrame 'df' is now guaranteed to exist from the merge.
# Test on first row only
df_test = df.iloc[[0]].copy()
df_test = df_test.apply(run_fal_generation, axis=1)
    
# Display results
print("\n" + "="*60)
print("📊 TEST RESULTS")
print("="*60)
    
result_cols = [
    'case_id', 
    'image_file_path',
    'ground_truth_3d_path', 
    'fal_generated_3d_path', 
    'fal_api_cost_usd', 
    'fal_api_token_count'
]
    
print(df_test[result_cols].to_string(index=False))
    
# Verify the generated file exists
if pd.notna(df_test.iloc[0]['fal_generated_3d_path']):
    gen_path = df_test.iloc[0]['fal_generated_3d_path']
    if os.path.exists(gen_path):
        size = os.path.getsize(gen_path) / 1024
        print(f"\n✅ Generated file verified: {gen_path} ({size:.1f} KB)")
    else:
        print(f"\n⚠️  Generated file path recorded but file not found: {gen_path}")
    
print("\n" + "="*60)
print("✅ TEST COMPLETE - Ready to run full batch if successful!")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.2/426.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.4/113.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.4/159.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.1

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [2]:
# CODE BLOCK 3: SIDE-BY-SIDE 3D MODEL COMPARISON (KAGGLE-COMPATIBLE)

import trimesh
import numpy as np
import os
import base64
from IPython.display import display, HTML

print("="*60)
print("🎨 3D MODEL COMPARISON VIEWER")
print("="*60)

# --- 1. Configuration ---
CASE_ID = 'S02_001'

DATASET_ROOT = '/kaggle/input/aird-benchmark-assets/'
OUTPUT_DIR = '/kaggle/working/results/'

GROUND_TRUTH_PATH = os.path.join(DATASET_ROOT, 'Data', f'{CASE_ID}_3DM.glb')
GENERATED_MODEL_PATH = os.path.join(OUTPUT_DIR, f'{CASE_ID}_FAL_GEN.glb')

# --- 2. Verify files exist ---
print("\n🔍 Checking file existence...")

if not os.path.exists(GROUND_TRUTH_PATH):
    raise FileNotFoundError(f"Ground truth not found: {GROUND_TRUTH_PATH}")
if not os.path.exists(GENERATED_MODEL_PATH):
    raise FileNotFoundError(f"Generated model not found: {GENERATED_MODEL_PATH}")

gt_size = os.path.getsize(GROUND_TRUTH_PATH) / 1024
gen_size = os.path.getsize(GENERATED_MODEL_PATH) / 1024

print(f"✅ Ground Truth: {gt_size:.1f} KB")
print(f"✅ Generated Model: {gen_size:.1f} KB")

# --- 3. Load models ---
print(f"\n📂 Loading 3D models...")

mesh_gt = trimesh.load_mesh(GROUND_TRUTH_PATH)
mesh_gen = trimesh.load_mesh(GENERATED_MODEL_PATH)

print(f"✅ Ground Truth: {len(mesh_gt.vertices):,} vertices, {len(mesh_gt.faces):,} faces")
print(f"✅ Generated: {len(mesh_gen.vertices):,} vertices, {len(mesh_gen.faces):,} faces")

# --- 4. Position side-by-side ---
print(f"\n🔄 Positioning models...")

shift_amount = mesh_gt.extents[0] * 1.5
translation_matrix = trimesh.transformations.translation_matrix([shift_amount, 0, 0])
mesh_gen.apply_transform(translation_matrix)

# Color coding
try:
    mesh_gt.visual.vertex_colors = [100, 100, 255, 255]  # Blue
    mesh_gen.visual.vertex_colors = [100, 255, 100, 255]  # Green
    print("✅ Color coded: Blue (GT) | Green (Generated)")
except:
    pass

# --- 5. Export combined scene ---
scene = trimesh.Scene([mesh_gt, mesh_gen])
COMPARISON_PATH = os.path.join(OUTPUT_DIR, f'{CASE_ID}_COMPARISON.glb')
scene.export(COMPARISON_PATH)
print(f"✅ Combined scene: {COMPARISON_PATH}")

# --- 6. Create inline viewer with embedded base64 GLB ---
print(f"\n🚀 Creating inline 3D viewer...")

# Read combined GLB and encode as base64
with open(COMPARISON_PATH, 'rb') as f:
    glb_base64 = base64.b64encode(f.read()).decode('utf-8')

print(f"✅ GLB encoded ({len(glb_base64)} chars)")

# Create fully embedded HTML with the GLB data inline
html_viewer = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.3.0/model-viewer.min.js"></script>
    <style>
        body {{
            margin: 0;
            padding: 20px;
            font-family: Arial, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        }}
        .container {{
            max-width: 1400px;
            margin: 0 auto;
            background: white;
            border-radius: 15px;
            padding: 30px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.3);
        }}
        h1 {{
            text-align: center;
            color: #333;
            margin: 0 0 15px 0;
        }}
        .legend {{
            text-align: center;
            margin-bottom: 25px;
            font-size: 18px;
        }}
        .legend span {{
            display: inline-block;
            margin: 0 15px;
            padding: 8px 20px;
            border-radius: 25px;
            font-weight: bold;
        }}
        .blue {{ background: #6666ff; color: white; }}
        .green {{ background: #66ff66; color: #222; }}
        model-viewer {{
            width: 100%;
            height: 650px;
            background: #2a2a2a;
            border-radius: 12px;
            border: 3px solid #ddd;
        }}
        .controls {{
            margin-top: 20px;
            padding: 20px;
            background: #f8f8f8;
            border-radius: 10px;
            text-align: center;
        }}
        .stats {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
            margin-top: 25px;
        }}
        .stat-box {{
            padding: 20px;
            background: #f0f0f0;
            border-radius: 10px;
            border-left: 5px solid #667eea;
        }}
        .stat-box h3 {{
            margin: 0 0 15px 0;
            color: #444;
        }}
        .stat-box p {{
            margin: 8px 0;
            color: #666;
            font-size: 15px;
        }}
        .highlight {{
            color: #667eea;
            font-weight: bold;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>🎨 3D Model Comparison: {CASE_ID}</h1>
        
        <div class="legend">
            <span class="blue">🔵 BLUE = Ground Truth (Original)</span>
            <span class="green">🟢 GREEN = FAL Generated (AI Output)</span>
        </div>
        
        <model-viewer 
            src="data:model/gltf-binary;base64,{glb_base64}"
            alt="Side-by-side 3D model comparison"
            auto-rotate
            rotation-per-second="30deg"
            camera-controls
            touch-action="pan-y"
            shadow-intensity="1"
            exposure="1.2"
            camera-orbit="45deg 75deg auto"
            field-of-view="35deg"
            min-camera-orbit="auto auto 10%"
            max-camera-orbit="auto auto 200%"
            interpolation-decay="100"
        ></model-viewer>
        
        <div class="controls">
            <p style="margin: 0; font-size: 16px; color: #555;">
                <strong>🖱️ Interactive Controls:</strong><br>
                Click & Drag = Rotate | Scroll = Zoom | Right-Click & Drag = Pan
            </p>
        </div>
        
        <div class="stats">
            <div class="stat-box">
                <h3>📊 Ground Truth (Original from Dataset)</h3>
                <p>📐 Vertices: <span class="highlight">{len(mesh_gt.vertices):,}</span></p>
                <p>🔺 Faces: <span class="highlight">{len(mesh_gt.faces):,}</span></p>
                <p>💾 File Size: <span class="highlight">{gt_size:.1f} KB</span></p>
                <p>📍 Position: <span class="highlight">Left (Blue)</span></p>
            </div>
            <div class="stat-box">
                <h3>🤖 FAL-Generated (AI Output)</h3>
                <p>📐 Vertices: <span class="highlight">{len(mesh_gen.vertices):,}</span></p>
                <p>🔺 Faces: <span class="highlight">{len(mesh_gen.faces):,}</span></p>
                <p>💾 File Size: <span class="highlight">{gen_size:.1f} KB</span></p>
                <p>📍 Position: <span class="highlight">Right (Green)</span></p>
                <p style="margin-top: 12px; font-weight: bold;">
                    ⚡ Complexity: <span class="highlight">{len(mesh_gen.vertices)/len(mesh_gt.vertices)*100:.1f}%</span> of original
                </p>
            </div>
        </div>
    </div>
</body>
</html>
"""

# Display the HTML directly in the notebook
display(HTML(html_viewer))

print("\n" + "="*60)
print("✅ INTERACTIVE 3D VIEWER DISPLAYED ABOVE!")
print("="*60)
print(f"📍 Layout:")
print(f"   🔵 LEFT  = Ground Truth (original from dataset)")
print(f"   🟢 RIGHT = FAL Generated (AI-created model)")
print("="*60)

# --- 7. Save standalone HTML file ---
HTML_PATH = os.path.join(OUTPUT_DIR, f'{CASE_ID}_VIEWER.html')
with open(HTML_PATH, 'w', encoding='utf-8') as f:
    f.write(html_viewer)

print(f"\n💾 Standalone viewer saved: {HTML_PATH}")
print(f"   Download this file to view in any browser")

# --- 8. Statistics ---
print(f"\n📊 Detailed Comparison:")
print(f"   Ground Truth: {len(mesh_gt.vertices):,} vertices | {len(mesh_gt.faces):,} faces")
print(f"   Generated:    {len(mesh_gen.vertices):,} vertices | {len(mesh_gen.faces):,} faces")
print(f"   Vertex Ratio: {len(mesh_gen.vertices)/len(mesh_gt.vertices)*100:.1f}%")
print(f"   Face Ratio:   {len(mesh_gen.faces)/len(mesh_gt.faces)*100:.1f}%")
print(f"   File Sizes:   GT={gt_size:.1f}KB | Gen={gen_size:.1f}KB")

print("\n" + "="*60)
print("✅ COMPARISON COMPLETE!")
print("="*60)

🎨 3D MODEL COMPARISON VIEWER

🔍 Checking file existence...
✅ Ground Truth: 1693.8 KB
✅ Generated Model: 1311.4 KB

📂 Loading 3D models...
✅ Ground Truth: 21,351 vertices, 30,442 faces
✅ Generated: 11,641 vertices, 16,652 faces

🔄 Positioning models...
✅ Color coded: Blue (GT) | Green (Generated)
✅ Combined scene: /kaggle/working/results/S02_001_COMPARISON.glb

🚀 Creating inline 3D viewer...
✅ GLB encoded (4102920 chars)



✅ INTERACTIVE 3D VIEWER DISPLAYED ABOVE!
📍 Layout:
   🔵 LEFT  = Ground Truth (original from dataset)
   🟢 RIGHT = FAL Generated (AI-created model)

💾 Standalone viewer saved: /kaggle/working/results/S02_001_VIEWER.html
   Download this file to view in any browser

📊 Detailed Comparison:
   Ground Truth: 21,351 vertices | 30,442 faces
   Generated:    11,641 vertices | 16,652 faces
   Vertex Ratio: 54.5%
   Face Ratio:   54.7%
   File Sizes:   GT=1693.8KB | Gen=1311.4KB

✅ COMPARISON COMPLETE!


In [3]:
# CODE BLOCK 4: WORKING DOWNLOAD SOLUTION

import os
import shutil
from IPython.display import display, HTML
import base64

print("="*60)
print("📥 CREATING DOWNLOADABLE FILES")
print("="*60)

# Source files
OUTPUT_DIR = '/kaggle/working/results/'
CASE_ID = 'S02_001'

HTML_PATH = os.path.join(OUTPUT_DIR, f'{CASE_ID}_VIEWER.html')
COMPARISON_PATH = os.path.join(OUTPUT_DIR, f'{CASE_ID}_COMPARISON.glb')
GENERATED_PATH = os.path.join(OUTPUT_DIR, f'{CASE_ID}_FAL_GEN.glb')

# Copy files to /kaggle/working/ (root level - Kaggle's download directory)
print("\n📂 Copying files to downloadable location...")

files_to_copy = {
    HTML_PATH: f'/kaggle/working/{CASE_ID}_VIEWER.html',
    COMPARISON_PATH: f'/kaggle/working/{CASE_ID}_COMPARISON.glb',
    GENERATED_PATH: f'/kaggle/working/{CASE_ID}_FAL_GEN.glb'
}

for src, dst in files_to_copy.items():
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size = os.path.getsize(dst) / 1024
        print(f"✅ Copied: {os.path.basename(dst)} ({size:.1f} KB)")
    else:
        print(f"❌ Source not found: {src}")

print("\n" + "="*60)
print("✅ FILES ARE NOW IN THE OUTPUT PANEL!")
print("="*60)

# Create a helpful guide
guide_html = f"""
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
            padding: 40px; border-radius: 20px; margin: 20px 0; color: white;">
    
    <h1 style="text-align: center; margin-bottom: 30px;">
        📥 How to Download Your 3D Models
    </h1>
    
    <div style="background: rgba(255,255,255,0.1); padding: 30px; border-radius: 15px; 
                backdrop-filter: blur(10px); margin-bottom: 25px;">
        <h2 style="margin-top: 0;">📍 STEP-BY-STEP INSTRUCTIONS:</h2>
        
        <div style="background: white; color: #333; padding: 25px; border-radius: 10px; 
                    margin: 20px 0; border-left: 5px solid #4CAF50;">
            <h3 style="margin-top: 0;">1️⃣ Look at the RIGHT sidebar</h3>
            <p style="margin: 10px 0; font-size: 16px;">
                Find the section labeled <strong>"Output (X.XMiB / XX.XGiB)"</strong>
            </p>
        </div>
        
        <div style="background: white; color: #333; padding: 25px; border-radius: 10px; 
                    margin: 20px 0; border-left: 5px solid #2196F3;">
            <h3 style="margin-top: 0;">2️⃣ Click on "/kaggle/working"</h3>
            <p style="margin: 10px 0; font-size: 16px;">
                Click the <strong>refresh icon (🔄)</strong> next to it if needed
            </p>
        </div>
        
        <div style="background: white; color: #333; padding: 25px; border-radius: 10px; 
                    margin: 20px 0; border-left: 5px solid #FF9800;">
            <h3 style="margin-top: 0;">3️⃣ You should now see these 3 files:</h3>
            <ul style="font-size: 16px; line-height: 1.8;">
                <li><strong>S02_001_VIEWER.html</strong> (4 MB) - Interactive viewer</li>
                <li><strong>S02_001_COMPARISON.glb</strong> (3 MB) - Combined models</li>
                <li><strong>S02_001_FAL_GEN.glb</strong> (1.3 MB) - AI-generated only</li>
            </ul>
        </div>
        
        <div style="background: white; color: #333; padding: 25px; border-radius: 10px; 
                    margin: 20px 0; border-left: 5px solid #9C27B0;">
            <h3 style="margin-top: 0;">4️⃣ Download the files</h3>
            <p style="margin: 10px 0; font-size: 16px;">
                Click the <strong>three dots (⋯)</strong> next to each filename → 
                Select <strong>"Download"</strong>
            </p>
        </div>
    </div>
    
    <div style="background: rgba(255,255,255,0.2); padding: 25px; border-radius: 15px;">
        <h2 style="margin-top: 0;">📖 What to do with the files:</h2>
        <div style="font-size: 16px; line-height: 1.8;">
            <p><strong>🌐 S02_001_VIEWER.html</strong></p>
            <p style="margin-left: 20px;">
                → Download and open in <strong>Chrome, Firefox, or Safari</strong><br>
                → You'll see BOTH models side-by-side in 3D with full rotation controls
            </p>
            
            <p style="margin-top: 20px;"><strong>📦 S02_001_COMPARISON.glb</strong></p>
            <p style="margin-left: 20px;">
                → Open in <strong>Blender</strong> or online at 
                <a href="https://gltf-viewer.donmccurdy.com/" target="_blank" 
                   style="color: #FFD700;">gltf-viewer.donmccurdy.com</a><br>
                → Contains both models (Blue + Green) positioned side-by-side
            </p>
            
            <p style="margin-top: 20px;"><strong>🤖 S02_001_FAL_GEN.glb</strong></p>
            <p style="margin-left: 20px;">
                → The AI-generated model only (without the original for comparison)
            </p>
        </div>
    </div>
    
    <div style="background: #FFD700; color: #333; padding: 20px; border-radius: 10px; 
                margin-top: 25px; font-weight: bold; text-align: center; font-size: 18px;">
        ⚠️ IMPORTANT: The files are NOW in /kaggle/working/ 
        (not in the results subfolder anymore)
    </div>
</div>
"""

display(HTML(guide_html))

# Show file tree
print("\n📁 File locations confirmed:")
print("   /kaggle/working/")
for filename in [f'{CASE_ID}_VIEWER.html', f'{CASE_ID}_COMPARISON.glb', f'{CASE_ID}_FAL_GEN.glb']:
    filepath = f'/kaggle/working/{filename}'
    if os.path.exists(filepath):
        size = os.path.getsize(filepath) / 1024
        print(f"   ├── {filename} ({size:.1f} KB)")

print("\n" + "="*60)
print("✅ ALL FILES READY FOR DOWNLOAD!")
print("="*60)
print("\n🎯 Go to the Output panel on the right →")
print("   Expand /kaggle/working/ →")
print("   Download the 3 files listed above")
print("="*60)

📥 CREATING DOWNLOADABLE FILES

📂 Copying files to downloadable location...
✅ Copied: S02_001_VIEWER.html (4011.0 KB)
✅ Copied: S02_001_COMPARISON.glb (3005.1 KB)
✅ Copied: S02_001_FAL_GEN.glb (1311.4 KB)

✅ FILES ARE NOW IN THE OUTPUT PANEL!



📁 File locations confirmed:
   /kaggle/working/
   ├── S02_001_VIEWER.html (4011.0 KB)
   ├── S02_001_COMPARISON.glb (3005.1 KB)
   ├── S02_001_FAL_GEN.glb (1311.4 KB)

✅ ALL FILES READY FOR DOWNLOAD!

🎯 Go to the Output panel on the right →
   Expand /kaggle/working/ →
   Download the 3 files listed above
